# 📊 全部 h5ad 文件对比 + 预处理检查清单

**目标**: 
1. 理解为什么 SRP182008、Mouse_Pancreas_1、Human_Pancreas_1、Bone_Marrow 的结构完全不同
2. 学会用"计算机视角"对每个 h5ad 做单元测试
3. 掌握针对不同来源数据的预处理策略

---

## 1. 为什么同一项目里的 h5ad 文件结构完全不一样?

**根本原因**: 它们来自完全不同的数据源, 经过不同的处理流程。

### 1.1 数据来源对比

| 数据集 | 来源 | 物种 | 处理工具 | obs 列风格 |
|--------|------|------|----------|-----------|
| **SRP182008** | NCBI GEO (SRA) | 拟南芥 | Seurat (R) 导出 | Seurat 风格 (Orig.ident, nCount_RNA, Percent.mt...) |
| **Mouse_Pancreas_1** | scvi-tools 导出 | 小鼠 | scanpy/scvi-tools | 简化 (cell_type, n_counts, size_factors) |
| **Human_Pancreas_1** | CELLxGENE Census | 人类 | CELLxGENE pipeline | CXG 标准化 (cell_type_ontology_term_id...) |
| **Bone_Marrow** | CELLxGENE Census | 人类 | CELLxGENE pipeline | CXG 标准化 |

### 1.2 obs 字段风格差异

**Seurat 风格 (SRP182008)**:
```
Orig.ident      ← 字符串, 来源标识
nCount_RNA      ← 浮点数, 总 counts
nFeature_RNA    ← 整数, 表达基因数
Percent.mt      ← 浮点数, 线粒体比例
Seurat_clusters ← 字符串, 聚类结果
Celltype        ← 字符串, 手动注释
```

**CXG 风格 (Bone_Marrow, Human_Pancreas_1)**:
```
cell_type                        ← 字符串
cell_type_ontology_term_id       ← ontology ID
n_counts                        ← 浮点数
size_factors                    ← 浮点数
raw_sum, nnz, raw_mean_nnz...  ← 统计量
```

> **为什么字段名不同?** 这是不同工具链导出的结果 — Seurat (R) 和 CELLxGENE (Python) 使用不同的命名惯例。preprocess.py 必须兼容这两种风格。

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp
import scanpy as sc

# NumPy 2.0 兼容性
if not hasattr(np, 'string_'):
    np.string_ = np.bytes_

DATA_DIR = '../data/'
OUTPUT_DIR = '../notebooks_figures/'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 加载所有数据集
print("加载数据中...")
adata_dict = {
    'SRP182008':        sc.read_h5ad(DATA_DIR + 'SRP182008.h5ad'),
    'Mouse_Pancreas_1': sc.read_h5ad(DATA_DIR + 'Mouse_Pancreas_1.h5ad'),
    'Human_Pancreas_1': sc.read_h5ad(DATA_DIR + 'Human_Pancreas_1.h5ad'),
    'Bone_Marrow':      sc.read_h5ad(DATA_DIR + 'Bone_Marrow.h5ad'),
}
print(f"✅ 加载了 {len(adata_dict)} 个数据集")

/data/luolie/conda/envs/scclubench-main/lib/python3.9/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


加载数据中...
✅ 加载了 4 个数据集


### 1.3 整体结构对比

In [2]:
# 整体对比表
rows = []
for name, adata in adata_dict.items():
    X = adata.X
    X_arr = X.toarray() if sp.issparse(X) else np.asarray(X)
    sparsity = 1 - X.nnz / (X.shape[0] * X.shape[1]) if sp.issparse(X) else 1 - np.count_nonzero(X_arr) / X_arr.size
    
    # 找 cell type 列
    ct_col = None
    for c in ['cell_type', 'Celltype', 'celltype']:
        if c in adata.obs.columns:
            ct_col = c
            break
    
    n_ct = adata.obs[ct_col].nunique() if ct_col else 0
    
    # 找 count 列
    count_col = None
    for c in ['nCount_RNA', 'n_counts', 'raw_sum']:
        if c in adata.obs.columns:
            count_col = c
            break
    
    median_counts = np.median(adata.obs[count_col].values) if count_col else np.nan
    
    rows.append({
        'Dataset': name,
        'Cells': f'{adata.n_obs:,}',
        'Genes': f'{adata.n_vars:,}',
        'X dtype': str(X.dtype),
        'Sparse': '✅' if sp.issparse(X) else '❌',
        'Sparsity': f'{sparsity:.1%}',
        'Integer counts': '✅' if np.allclose(X_arr, X_arr.astype(int), atol=1e-3) else '⚠️',
        'Celltype col': ct_col or '❌ 无',
        'Celltypes': n_ct,
        'Median counts': f'{median_counts:,.0f}' if not np.isnan(median_counts) else 'N/A',
        'Has raw': '✅' if adata.raw is not None else '❌',
        'Has layers': '✅' if adata.layers else '❌',
        'n obs columns': len(adata.obs.columns),
        'n var columns': len(adata.var.columns),
    })

comparison_df = pd.DataFrame(rows)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
print("📊 所有数据集整体对比:")
display(comparison_df.T)

📊 所有数据集整体对比:


,0,1,2,3
Dataset,SRP182008,Mouse_Pancreas_1,Human_Pancreas_1,Bone_Marrow
Cells,"13,514","1,886","2,544","8,357"
Genes,"53,678","14,878","61,497","61,497"
X dtype,float64,float32,int64,float32
Sparse,✅,✅,✅,✅
Sparsity,97.6%,89.0%,94.9%,99.8%
Integer counts,✅,✅,✅,✅
Celltype col,Celltype,cell_type,cell_type,cell_type
Celltypes,15,13,7,35
Median counts,"1,890","4,251","443,334","1,665"


### 1.4 obs 列名差异的可视化

In [3]:
# 绘制 obs 列名对比图
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, (name, adata) in enumerate(adata_dict.items()):
    ax = axes[i]
    cols = list(adata.obs.columns)
    colors = ['#2ecc71' if 'count' in c.lower() or 'feature' in c.lower() 
              else '#3498db' if 'type' in c.lower() or 'cell' in c.lower()
              else '#9b59b6' if c in ['batch', 'Dataset', 'dataset']
              else '#e74c3c' if 'cluster' in c.lower()
              else '#95a5a6' for c in cols]
    
    ax.barh(range(len(cols)), [1]*len(cols), color=colors)
    ax.set_yticks(range(len(cols)))
    ax.set_yticklabels(cols, fontsize=8)
    ax.set_xlim(0, 1)
    ax.set_title(f'{name}
({len(cols)} obs columns)', fontsize=11)
    ax.set_xticks([])

# 图例
legend_elements = [
    plt.Rectangle((0,0),1,1, facecolor='#2ecc71', label='QC metric (count/feature)'),
    plt.Rectangle((0,0),1,1, facecolor='#3498db', label='Cell identity (type/tissue)'),
    plt.Rectangle((0,0),1,1, facecolor='#9b59b6', label='Batch/Dataset'),
    plt.Rectangle((0,0),1,1, facecolor='#e74c3c', label='Cluster result'),
    plt.Rectangle((0,0),1,1, facecolor='#95a5a6', label='Other'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=5, fontsize=9,
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle('obs 字段对比: 不同来源数据的字段风格差异', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'h5ad_comparison_obs_columns.png', dpi=150, bbox_inches='tight')
plt.show()

SyntaxError: EOL while scanning string literal (2096294563.py, line 18)

---

## 2. X 矩阵状态对比

**关键问题**: 每个数据集的 X 处于什么状态 — 原始 counts? 已经归一化? 已经 log1p?

In [ ]:
print("=" * 70)
print("🔍 X 矩阵状态检测 — 这是预处理前最重要的检查")
print("=" * 70)

for name, adata in adata_dict.items():
    print(f"\n{'─'*60}")
    print(f"📁 {name}")
    print(f"{'─'*60}")
    X = adata.X
    
    # 1. 稀疏性
    if sp.issparse(X):
        X_sample = X.data[:200_000]
        nnz_info = f"sparse CSR, nnz={X.nnz:,}"
    else:
        X_arr = np.asarray(X)
        X_sample = X_arr.flatten()[:200_000]
        nnz_info = f"dense {X.shape}"
    
    # 2. 数据类型
    print(f"  dtype: {X.dtype}")
    
    # 3. 数值范围
    print(f"  范围:  [{X_sample.min():.4f}, {X_sample.max():.4f}]")
    print(f"  均值:   {X_sample.mean():.4f}")
    print(f"  中位数: {np.median(X_sample):.4f}")
    
    # 4. 整数性检验 (原始 counts 的标志)
    is_int = np.allclose(X_sample, X_sample.astype(int), atol=1e-3)
    int_frac = np.mean(np.abs(X_sample - X_sample.astype(int)) < 1e-3)
    print(f"  接近整数: {is_int} ({int_frac:.1%} 的值)")
    
    # 5. 是否 float32/int64 暗示已经过某种处理
    if X.dtype == np.float32 or X.dtype == np.float64:
        if is_int:
            print(f"  → 判断: float 但接近整数 = **原始 Counts** (正常)")
        elif X_sample.max() < 20:
            print(f"  → 判断: float 且最大值 < 20 = **可能已经 log1p** ⚠️")
        else:
            print(f"  → 判断: float 且非整数 = **可能已经归一化** ⚠️")
    elif X.dtype in [np.int32, np.int64]:
        print(f"  → 判断: 整数类型 = **原始 Counts** (正常)")
    
    # 6. raw 检查
    if adata.raw is not None:
        raw_sample = adata.raw.X.data[:100_000] if sp.issparse(adata.raw.X) else adata.raw.X.flatten()[:100_000]
        raw_is_int = np.allclose(raw_sample, raw_sample.astype(int), atol=1e-3)
        print(f"  raw X 是整数: {raw_is_int}")

In [ ]:
# 可视化 X 分布对比
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, (name, adata) in enumerate(adata_dict.items()):
    X = adata.X
    if sp.issparse(X):
        data = X.data[:300_000]
    else:
        data = np.asarray(X).flatten()[:300_000]
    
    # 原始分布
    axes[i].hist(data, bins=100, color='steelblue', edgecolor='none', alpha=0.8)
    axes[i].set_title(f'{name}
原始分布', fontsize=10)
    axes[i].set_xlabel('Expression')
    axes[i].set_ylabel('Frequency')
    
    # log1p 分布
    log_data = np.log1p(data)
    axes[i+4].hist(log_data, bins=100, color='coral', edgecolor='none', alpha=0.8)
    axes[i+4].set_title(f'{name}
log1p 分布', fontsize=10)
    axes[i+4].set_xlabel('log1p(Expression)')
    axes[i+4].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'h5ad_comparison_x_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 3. 预处理检查清单 (Preprocessing Checklist)

**每个数据集在预处理前, 必须逐一验证以下项目。** 这不是"可选"的 — 这是数据处理中的 assert 语句。

In [ ]:
def run_preprocessing_checklist(name, adata):
    """
    对单个数据集运行预处理前检查清单
    返回: (passed, issues)
    """
    issues = []
    passed = []
    
    X = adata.X
    if sp.issparse(X):
        X_data = X.data[:500_000]
    else:
        X_data = np.asarray(X).flatten()[:500_000]
    
    print(f"\n{'='*60}")
    print(f"  🩺 预处理前检查清单: {name}")
    print(f"{'='*60}")
    
    # ── 基础信息 ────────────────────────────────────────────
    print(f"\n  【基础信息】")
    print(f"    ✅ 细胞数:   {adata.n_obs:,}")
    print(f"    ✅ 基因数:   {adata.n_vars:,}")
    print(f"    ✅ X 类型:   {type(X).__name__} ({X.dtype})")
    
    # ── 1. 整数性检查 ──────────────────────────────────────
    print(f"\n  【1/12] 整数性检查 (X 是否为原始 Counts)]")
    is_int = np.allclose(X_data, X_data.astype(int), atol=1e-3)
    int_frac = np.mean(np.abs(X_data - X_data.astype(int)) < 1e-3)
    print(f"    {'✅' if is_int else '⚠️'} {int_frac:.1%} 的值接近整数")
    if is_int:
        print(f"    → 结论: X 是原始 Counts ✓")
        passed.append("X 是原始 Counts")
    else:
        print(f"    → 结论: X 可能已处理 (归一化? log1p?) ⚠️")
        issues.append("X 可能已处理, 需要确认")
    
    # ── 2. 稀疏度 ──────────────────────────────────────────
    print(f"\n  【2/12] 稀疏度检查")
    if sp.issparse(X):
        sparsity = 1 - X.nnz / (X.shape[0] * X.shape[1])
    else:
        sparsity = 1 - np.count_nonzero(X_data) / len(X_data)
    print(f"    📊 稀疏度: {sparsity:.2%}")
    if sparsity > 0.5:
        print(f"    → 高稀疏度 scRNA-seq 典型特征 ✓")
    else:
        print(f"    ⚠️ 稀疏度偏低, 可能来自 Smart-seq2 等富集方法")
    
    # ── 3. Celltype 列 ─────────────────────────────────────
    print(f"\n  【3/12] Celltype 列检查")
    ct_col = None
    for c in ['cell_type', 'Celltype', 'celltype', 'cell_label']:
        if c in adata.obs.columns:
            ct_col = c
            break
    if ct_col:
        n_ct = adata.obs[ct_col].nunique()
        null_ct = adata.obs[ct_col].isnull().sum()
        print(f"    ✅ 找到 celltype 列: '{ct_col}'")
        print(f"    ✅ 细胞类型数: {n_ct}")
        print(f"    {'✅' if null_ct == 0 else '⚠️'} 空值数: {null_ct}")
        passed.append(f"Celltype 列存在: {ct_col}")
        
        # 检查 Unknown 类别
        if 'Unknow' in adata.obs[ct_col].values or 'unknown' in adata.obs[ct_col].values:
            unk = adata.obs[ct_col].value_counts().get('Unknow', 0) + adata.obs[ct_col].value_counts().get('unknown', 0)
            print(f"    ⚠️ 存在 Unknown 类别: {unk} cells (建议聚类时过滤或单独分析)")
            issues.append(f"存在 {unk} 个 Unknown 细胞")
    else:
        print(f"    ❌ 未找到 celltype 列!")
        print(f"    可用列: {list(adata.obs.columns)}")
        issues.append("缺少 celltype 列")
    
    # ── 4. Batch 列 ───────────────────────────────────────
    print(f"\n  【4/12] Batch 列检查")
    batch_col = None
    for c in ['batch', 'Dataset', 'dataset_id', 'donor_id']:
        if c in adata.obs.columns:
            batch_col = c
            break
    if batch_col:
        n_batch = adata.obs[batch_col].nunique()
        print(f"    ✅ 找到 batch 列: '{batch_col}', 批次数: {n_batch}")
        if n_batch > 1:
            print(f"    → 存在批次效应, 需要考虑批次校正")
            issues.append("存在批次效应")
        else:
            print(f"    → 单批次数据, 无需批次校正")
    else:
        print(f"    ⚠️ 未找到明确的 batch 列, 无法做批次校正")
    
    # ── 5. QC 列 ──────────────────────────────────────────
    print(f"\n  【5/12] QC 指标列检查")
    count_col = None
    for c in ['nCount_RNA', 'n_counts', 'raw_sum']:
        if c in adata.obs.columns:
            count_col = c
            break
    if count_col:
        vals = adata.obs[count_col].values
        print(f"    ✅ 找到 count 列: '{count_col}'")
        print(f"    → 范围: [{vals.min():.1f}, {vals.max():.1f}], 中位数: {np.median(vals):.1f}")
        passed.append("QC count 列存在")
    else:
        print(f"    ❌ 未找到 count 列, 需要手动计算")
        issues.append("缺少 QC count 列")
    
    # ── 6. raw 数据 ────────────────────────────────────────
    print(f"\n  【6/12] raw 数据检查")
    if adata.raw is not None:
        print(f"    ✅ raw 存在, shape={adata.raw.shape}")
        if adata.raw.n_vars == adata.n_vars:
            print(f"    → raw 基因数=当前基因数, 未经基因过滤")
        else:
            print(f"    → raw 基因数({adata.raw.n_vars}) > 当前({adata.n_vars}), 已过滤基因")
        passed.append("raw 数据存在")
    else:
        print(f"    ⚠️ raw 不存在 — 预处理后将无法恢复原始数据")
        issues.append("缺少 raw 备份")
    
    # ── 7. layers ─────────────────────────────────────────
    print(f"\n  【7/12] layers 检查")
    if adata.layers:
        print(f"    ✅ 存在 layers: {list(adata.layers.keys())}")
        passed.append("layers 存在")
    else:
        print(f"    ⚠️ layers 为空, 无预存 counts/norm_log")
        print(f"    → preprocess.py 需要自己计算这些")
        issues.append("缺少预存 layers")
    
    # ── 8. 零行零列 ──────────────────────────────────────
    print(f"\n  【8/12] 全零行/列检查")
    # 检查全零行
    if sp.issparse(X):
        row_sums = np.array(X.sum(axis=1)).flatten()
        zero_rows = (row_sums == 0).sum()
        col_sums = np.array(X.sum(axis=0)).flatten()
        zero_cols = (col_sums == 0).sum()
    else:
        X_arr = np.asarray(X)
        zero_rows = (X_arr.sum(axis=1) == 0).sum()
        zero_cols = (X_arr.sum(axis=0) == 0).sum()
    
    print(f"    {'✅' if zero_rows == 0 else '⚠️'} 全零行(细胞): {zero_rows}")
    print(f"    {'✅' if zero_cols == 0 else '⚠️'} 全零列(基因): {zero_cols}")
    if zero_rows > 0 or zero_cols > 0:
        issues.append(f"存在 {zero_rows} 个全零行, {zero_cols} 个全零列")
    else:
        passed.append("无全零行/列")
    
    # ── 9. 异常值 ────────────────────────────────────────
    print(f"\n  【9/12] 异常值检查")
    max_val = X_data.max()
    # 原始 counts 通常不超过 1e6
    if is_int and max_val > 1e6:
        print(f"    ⚠️ 最大值 {max_val:.0f} 异常大, 可能是异常细胞")
        issues.append(f"存在异常表达值: max={max_val:.0f}")
    else:
        print(f"    ✅ 最大值 {max_val:.2f} 在合理范围内")
    
    # ── 10. 基因名格式 ────────────────────────────────────
    print(f"\n  【10/12] 基因名格式检查")
    gene_sample = list(adata.var.index[:5])
    print(f"    基因名示例: {gene_sample}")
    if gene_sample[0].startswith('AT'):
        print(f"    → 物种: 拟南芥 (Arabidopsis thaliana)")
        passed.append("基因名格式: 拟南芥 AT 格式")
    elif all(g.isdigit() for g in gene_sample):
        print(f"    → 物种: 未知 (数字索引, 非标准基因名)")
        issues.append("基因名是数字索引, 非标准符号")
    elif 'ENSG' in str(gene_sample[0]):
        print(f"    → 物种: 人类 (ENSG 基因 ID)")
        passed.append("基因名格式: ENSG 人类格式")
    elif 'ENSMUSG' in str(gene_sample[0]):
        print(f"    → 物种: 小鼠 (ENSMUSG 基因 ID)")
        passed.append("基因名格式: ENSMUSG 小鼠格式")
    else:
        print(f"    → 无法确定物种")
    
    # ── 11. 内存占用 ──────────────────────────────────────
    print(f"\n  【11/12] 内存占用估算")
    n, d = X.shape
    dense_mb = n * d * 8 / 1e6
    print(f"    若存为 dense (float64): ~{dense_mb:.0f} MB")
    print(f"    当前基因数: {d:,}, 细胞数: {n:,}")
    if dense_mb > 5000:
        print(f"    ⚠️ 建议使用 scanpy 处理 (稀疏矩阵 + 分批计算)")
        issues.append("数据集较大, 需注意内存")
    
    # ── 12. 重复基因名 ───────────────────────────────────
    print(f"\n  【12/12] 重复基因名检查")
    dup_genes = adata.var.index.duplicated().sum()
    print(f"    重复基因名: {dup_genes}")
    if dup_genes > 0:
        print(f"    ⚠️ 存在重复基因名, 需要去重")
        issues.append("存在重复基因名")
    else:
        print(f"    ✅ 无重复基因名")
        passed.append("无重复基因名")
    
    # ── 总结 ──────────────────────────────────────────────
    print(f"\n  {'='*60}")
    print(f"  📋 检查结果汇总: {name}")
    print(f"  {'='*60}")
    print(f"  ✅ 通过: {len(passed)} 项")
    for p in passed:
        print(f"     • {p}")
    if issues:
        print(f"\n  ⚠️ 问题: {len(issues)} 项")
        for issue in issues:
            print(f"     • {issue}")
    else:
        print(f"\n  ✅ 无严重问题, 可以进行预处理!")
    
    return passed, issues

# 对所有数据集运行检查
all_passed = {}
all_issues = {}
for name, adata in adata_dict.items():
    p, i = run_preprocessing_checklist(name, adata)
    all_passed[name] = p
    all_issues[name] = i

---

## 4. 预处理后验证 (Post-preprocessing Checks)

**在 preprocess.py 运行后, 必须验证以下项目, 确认数据没有出错。**

In [ ]:
def run_post_preprocessing_checks(name, adata, label_col='cell_type'):
    """
    模拟预处理后的检查
    在实际使用 preprocess.py 后, 运行此检查
    """
    print(f"\n{'='*60}")
    print(f"  🔬 预处理后检查: {name}")
    print(f"{'='*60}")
    
    # 1. X shape
    print(f"\n  [1/8] X 维度检查")
    print(f"    形状: {adata.shape}")
    if adata.n_vars == 1000:
        print(f"    ✅ 正确: 1000 个 HVG (preprocess.py 默认行为)")
    elif adata.n_vars < 100:
        print(f"    ⚠️ 基因数偏少, 可能影响模型性能")
    elif adata.n_vars > 5000:
        print(f"    ⚠️ 基因数偏多, HVG 筛选可能未生效")
    
    # 2. size_factors
    print(f"\n  [2/8] size_factors 检查")
    if 'size_factors' in adata.obs.columns:
        sf = adata.obs['size_factors'].values
        print(f"    ✅ 存在, 范围: [{sf.min():.3f}, {sf.max():.3f}]")
        if sf.min() < 0:
            print(f"    ⚠️ size_factors 有负值, 异常!")
    else:
        print(f"    ⚠️ size_factors 不存在 (preprocess.py 中 size_factors=True 时才生成)")
    
    # 3. layers['norm_log']
    print(f"\n  [3/8] layers['norm_log'] 检查")
    if 'norm_log' in adata.layers:
        nl = adata.layers['norm_log']
        print(f"    ✅ 存在, shape: {nl.shape}")
        nl_flat = nl.flatten()[:500_000] if sp.issparse(nl) else nl.toarray().flatten()[:500_000]
        print(f"    ✅ 范围: [{nl_flat.min():.4f}, {nl_flat.max():.4f}]")
        print(f"    ✅ 均值: {nl_flat.mean():.4f}")
        if np.all(nl_flat >= 0):
            print(f"    ✅ norm_log 非负, 符合 log1p 输出")
        else:
            print(f"    ⚠️ norm_log 有负值, 可能是 scale 后存入")
    else:
        print(f"    ⚠️ layers['norm_log'] 不存在")
    
    # 4. X 标准化检查
    print(f"\n  [4/8] X 分布检查 (Z-score vs norm_log)")
    X = adata.X
    X_flat = X.data[:500_000] if sp.issparse(X) else np.asarray(X).flatten()[:500_000]
    x_mean = X_flat.mean()
    x_std = X_flat.std()
    print(f"    X 均值: {x_mean:.4f}")
    print(f"    X 标准差: {x_std:.4f}")
    
    is_scaled = abs(x_mean) < 0.1 and abs(x_std - 1.0) < 0.2
    is_log_like = x_flat.max() < 20 and np.all(x_flat >= 0)
    
    if is_scaled:
        print(f"    ✅ X 接近 Z-score 标准化 (均值≈0, std≈1)")
        print(f"    → 这意味着 X 已用于神经网络训练 (preprocess.py normalize_input=True)")
    elif is_log_like:
        print(f"    ✅ X 接近 log1p 分布 (max<20, 非负)")
        print(f"    → 这意味着 X 是 norm_log 数据 (normalize_input=False)")
    else:
        print(f"    ⚠️ X 分布不明确, 可能是混合状态")
    
    # 5. 标签检查
    print(f"\n  [5/8] 细胞类型标签检查")
    if label_col in adata.obs.columns:
        n_ct = adata.obs[label_col].nunique()
        null_ct = adata.obs[label_col].isnull().sum()
        print(f"    ✅ '{label_col}' 存在, {n_ct} 种类型")
        print(f"    {'✅' if null_ct == 0 else '⚠️'} 空值: {null_ct}")
        if null_ct > 0:
            print(f"    ⚠️ 存在空标签, 评估时需过滤")
    else:
        print(f"    ❌ '{label_col}' 不存在!")
    
    # 6. 细胞数量一致性
    print(f"\n  [6/8] 细胞数量一致性")
    print(f"    X 细胞数: {adata.n_obs:,}")
    print(f"    obs 行数: {len(adata.obs):,}")
    if adata.n_obs == len(adata.obs):
        print(f"    ✅ 一致")
    else:
        print(f"    ❌ 不一致!")
    
    # 7. raw 一致性
    print(f"\n  [7/8] raw 数据一致性")
    if adata.raw is not None:
        if adata.raw.n_obs == adata.n_obs:
            print(f"    ✅ raw 细胞数=当前细胞数, 无细胞被过滤")
        else:
            print(f"    ⚠️ raw 细胞数 ({adata.raw.n_obs}) ≠ 当前 ({adata.n_obs})")
    
    # 8. NaN/Inf 检查
    print(f"\n  [8/8] NaN/Inf 检查")
    X = adata.X
    if sp.issparse(X):
        has_nan = np.any(np.isnan(X.data))
        has_inf = np.any(np.isinf(X.data))
    else:
        has_nan = np.any(np.isnan(X.toarray()))
        has_inf = np.any(np.isinf(X.toarray()))
    
    print(f"    {'❌ 有 NaN!' if has_nan else '✅ 无 NaN'}")
    print(f"    {'❌ 有 Inf!' if has_inf else '✅ 无 Inf'}")
    if has_nan or has_inf:
        print(f"    ⚠️ 数据异常, 必须修复后才能进行聚类!")

# 演示预处理后检查 (使用 SRP182008)
print("=" * 70)
print("⚠️  注意: 以下演示使用原始数据, 预处理后应重新运行此检查")
print("=" * 70)
run_post_preprocessing_checks('SRP182008 (预处理前)', adata_dict['SRP182008'])

---

## 5. preprocess.py 的兼容性测试

**核心问题**: preprocess.py 能否正确处理所有数据集? 它的 `normalize_sc()` 函数依赖特定列名。让我验证。

In [ ]:
# 测试 preprocess.py 能否找到所需列
print("=" * 70)
print("🔧 preprocess.py 兼容性测试")
print("=" * 70)

# size_factors 计算需要的列
count_col_candidates = ['n_counts', 'total_counts', 'nCount_RNA', 'total']

# label 列
label_col_candidates = ['cell_type', 'Celltype', 'celltype', 'cell_label', 'label']

rows = []
for name, adata in adata_dict.items():
    # 找 count 列
    found_count = None
    for c in count_col_candidates:
        if c in adata.obs.columns:
            found_count = c
            break
    
    # 找 label 列
    found_label = None
    for c in label_col_candidates:
        if c in adata.obs.columns:
            found_label = c
            break
    
    # 找 batch 列
    found_batch = None
    for c in ['batch', 'Dataset', 'dataset_id']:
        if c in adata.obs.columns:
            found_batch = c
            break
    
    rows.append({
        'Dataset': name,
        'Count 列': found_count or '❌ 未找到',
        'Label 列': found_label or '❌ 未找到',
        'Batch 列': found_batch or '⚠️ 未找到',
    })

compat_df = pd.DataFrame(rows)
display(compat_df)

print("\n  ✅ 总结:")
print(f"    所有数据集都能找到 count 列 — normalize_sc() 可以正常工作")
print(f"    所有数据集都能找到 label 列 — prepare_data_for_model() 可以正常工作")
print(f"    SRP182008 只有 1 个 Dataset, 无批次效应")
print(f"    Human_Pancreas_1 和 Bone_Marrow 来自 CXG, 可能有 donor_id 可作为批次")

---

## 6. 常见陷阱汇总

### 6.1 不同 h5ad 的陷阱

| 陷阱 | 影响 | 检测方法 |
|------|------|---------|
| `np.string_` 在 NumPy 2.0 报错 | h5ad 无法读取 | 加 `np.string_ = np.bytes_` 补丁 |
| 基因名是数字索引 | 无法与 GeneVocab 匹配 | 检查 var.index 格式 |
| X 已是 float32 但接近整数 | 可能是已归一化的 counts | 检查 is_integer 断言 |
| 无 size_factors 列 | scVI 等模型无法工作 | 检查 obs 列 |
| Celltype 列名不统一 | prepare_data_for_model() 报错 | 统一映射 |
| 重复基因名 | HVG 筛选后索引错位 | var.index.duplicated() 检查 |

### 6.2 预处理陷阱 (preprocess.py)

| 陷阱 | 危害 | 修复方法 |
|------|------|---------|
| 二次 log1p | 数据过度变换, 极端压缩 | check_normalization() 检测 |
| 二次 scale | 数据完全失真 | 检查 is_scaled 断言 |
| scVI 使用 z-score 数据 | scVI 需要原始 counts | scVI 单独处理 |
| HVG 后标签错位 | 所有下游分析全错 | 始终在 subset 后的 adata 上操作 |
| 未知细胞类型参与聚类评估 | 评估指标偏低 | 过滤 Unknown 细胞 |

### 6.3 跨模型陷阱

| 陷阱 | 危害 | 修复方法 |
|------|------|---------|
| 不同模型用不同细胞 | embedding 对不上 | 记录 cell 顺序 |
| 不同模型用不同基因 | embedding 维度不同 | 统一 HVG 列表 |
| 选错 batch 列 | 把 Dataset 当 batch | 检查 batch 列的唯一值数 |
| 混用 norm_log 和 z-score | 模型输入不一致 | layers['norm_log'] 保存原始 norm |

### 6.4 SRP182008 特殊注意事项

1. **Celltype = "Unknow"**: 聚类前建议过滤, 避免干扰评估
2. **Percent.mt 列**: 拟南芥线粒体基因命名非标准, 此列可能无效
3. **X dtype = float64**: Seurat 导出时转为 float, 但值仍接近整数
4. **只有 1 个 Dataset**: 无需批次校正
5. **Seurat_clusters 有 24 个**: 比 Celltype (15) 更细, 可作为参考

In [ ]:
# 最终总结可视化
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 图1: 细胞数 vs 基因数
for name, adata in adata_dict.items():
    axes[0].scatter(adata.n_vars, adata.n_obs, s=200, label=name, alpha=0.8)
    axes[0].annotate(name, (adata.n_vars, adata.n_obs), 
                     textcoords='offset points', xytext=(5, 5), fontsize=9)
axes[0].set_xlabel('Number of genes')
axes[0].set_ylabel('Number of cells')
axes[0].set_title('Cells × Genes (对数尺度)')
axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

# 图2: 细胞类型数
ct_counts = {}
for name, adata in adata_dict.items():
    for c in ['cell_type', 'Celltype', 'celltype']:
        if c in adata.obs.columns:
            ct_counts[name] = adata.obs[c].nunique()
            break

names = list(ct_counts.keys())
counts = list(ct_counts.values())
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']
axes[1].bar(names, counts, color=colors)
axes[1].set_ylabel('Cell type count')
axes[1].set_title('Ground Truth Cell Types')
for i, (n, c) in enumerate(zip(names, counts)):
    axes[1].text(i, c + 0.3, str(c), ha='center', fontsize=11)

# 图3: 稀疏度
sparsities = {}
for name, adata in adata_dict.items():
    X = adata.X
    if sp.issparse(X):
        s = 1 - X.nnz / (X.shape[0] * X.shape[1])
    else:
        X_arr = np.asarray(X)
        s = 1 - np.count_nonzero(X_arr) / X_arr.size
    sparsities[name] = s

names = list(sparsities.keys())
spars = [sparsities[n] * 100 for n in names]
axes[2].bar(names, spars, color=colors)
axes[2].set_ylabel('Sparsity (%)')
axes[2].set_title('Data Sparsity')
for i, (n, s) in enumerate(zip(names, spars)):
    axes[2].text(i, s + 0.5, f'{s:.1f}%', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'h5ad_comparison_summary.png', dpi=150, bbox_inches='tight')
plt.show()